In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict, cross_val_score, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

# Gradient Boosting variants
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Linear models
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Neural network
from sklearn.neural_network import MLPRegressor

# Stacking
from sklearn.ensemble import StackingRegressor


def parity_plot(y_true, y_pred, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
    
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    ax.scatter(y_true, y_pred, alpha=0.5, color='#2E86AB')
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 
            'k--', lw=2, label='Perfect prediction')
    
    metrics_text = f'R² = {r2:.3f}\nRMSE = {rmse:.3f}\nMAE = {mae:.3f}'
    ax.text(0.05, 0.95, metrics_text, 
            transform=ax.transAxes,
            fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Actual', fontsize=12, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    return r2, rmse, mae


def compare_models_oof(X, y, cv=5, scale_features=False):
    # Optional feature scaling
    if scale_features:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = X
    
    # Define models to test
    models = {
        'Gradient Boosting': GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            min_samples_leaf=4,
            subsample=0.8,
            random_state=1065
        ),
        'XGBoost': XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=1065,
            tree_method='hist'
        ),
        'LightGBM': LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=1065,
            verbose=-1
        ),
        'CatBoost': CatBoostRegressor(
            iterations=300,
            learning_rate=0.05,
            depth=5,
            subsample=0.8,
            random_state=1065,
            verbose=False
        ),
        'Random Forest': RandomForestRegressor(
            n_estimators=300,
            max_depth=15,
            min_samples_leaf=4,
            random_state=1065,
            n_jobs=-1
        ),
        'Extra Trees': ExtraTreesRegressor(
            n_estimators=300,
            max_depth=15,
            min_samples_leaf=4,
            random_state=1065,
            n_jobs=-1
        ),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1, max_iter=5000),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
        'Neural Network': MLPRegressor(
            hidden_layer_sizes=(100, 50),
            activation='relu',
            max_iter=10000,
            random_state=1065,
            early_stopping=True
        )
    }
    
    results = []
    
    for name, model in models.items():
        print(f"\nEvaluating {name}...")
        
        # Use scaled or unscaled data
        if name in ['Ridge', 'Lasso', 'ElasticNet', 'Neural Network']:
            X_use = X_scaled
        else:
            X_use = X
        
        # Get out-of-fold predictions
        try:
            oof_pred = cross_val_predict(model, X_use, y, cv=cv, n_jobs=-1)
        except Exception as e:
            print(f"  Error with {name}: {e}")
            continue
        
        # Calculate OOF metrics
        oof_r2 = r2_score(y, oof_pred)
        oof_rmse = np.sqrt(mean_squared_error(y, oof_pred))
        oof_mae = mean_absolute_error(y, oof_pred)
        
        # Get CV scores for comparison
        cv_scores = cross_val_score(model, X_use, y, cv=cv, scoring='r2', n_jobs=-1)
        cv_r2_mean = cv_scores.mean()
        cv_r2_std = cv_scores.std()
        
        results.append({
            'Model': name,
            'OOF R²': oof_r2,
            'CV R² (mean)': cv_r2_mean,
            'CV R² (std)': cv_r2_std,
            'OOF RMSE': oof_rmse,
            'OOF MAE': oof_mae,
        })
        
    # Sort by OOF R²
    results_sorted = sorted(results, key=lambda x: x['OOF R²'], reverse=True)
    return results_sorted

def hyperparameter_tune_model_oof(X, y, model_name, cv=5, scale_features=False):
    # Optional feature scaling
    if scale_features:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = X
    
    # Use scaled data for linear models and neural networks
    if model_name in ['Ridge', 'Lasso', 'ElasticNet', 'Neural Network']:
        X_use = X_scaled
    else:
        X_use = X
    
    # Define model and parameter grid based on model name
    if model_name == 'LightGBM':
        model = LGBMRegressor(random_state=1065, verbose=-1)
        param_grid = {
            'n_estimators': [200, 300, 500, 700],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'max_depth': [4, 5, 6, 7, 8],
            'num_leaves': [20, 31, 40, 50],
            'min_child_samples': [10, 20, 30, 40],
            'subsample': [0.7, 0.8, 0.9],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1.0],
            'reg_lambda': [0, 0.01, 0.1, 1.0]
        }
        n_iter = 100
    
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=1065, tree_method='hist')
        param_grid = {
            'n_estimators': [200, 300, 500, 700],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'max_depth': [4, 5, 6, 7, 8],
            'min_child_weight': [1, 3, 5, 7],
            'subsample': [0.7, 0.8, 0.9],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'gamma': [0, 0.1, 0.5, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1.0],
            'reg_lambda': [0, 0.01, 0.1, 1.0]
        }
        n_iter = 100
    
    elif model_name == 'CatBoost':
        model = CatBoostRegressor(random_state=1065, verbose=False)
        param_grid = {
            'iterations': [200, 300, 500, 700],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'depth': [4, 5, 6, 7, 8],
            'l2_leaf_reg': [1, 3, 5, 7, 9],
            'subsample': [0.7, 0.8, 0.9],
            'colsample_bylevel': [0.7, 0.8, 0.9, 1.0]
        }
        n_iter = 100
    
    elif model_name == 'Gradient Boosting':
        model = GradientBoostingRegressor(random_state=1065)
        param_grid = {
            'n_estimators': [200, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 4, 5, 6],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [2, 4, 6],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'max_features': ['sqrt', 'log2', None]
        }
        n_iter = 80
    
    elif model_name == 'Random Forest':
        model = RandomForestRegressor(random_state=1065, n_jobs=-1)
        param_grid = {
            'n_estimators': [200, 300, 500],
            'max_depth': [10, 15, 20, 25, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
            'bootstrap': [True, False]
        }
        n_iter = 80
    
    elif model_name == 'Extra Trees':
        model = ExtraTreesRegressor(random_state=1065, n_jobs=-1)
        param_grid = {
            'n_estimators': [200, 300, 500],
            'max_depth': [10, 15, 20, 25, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
            'bootstrap': [True, False]
        }
        n_iter = 80
    
    elif model_name == 'Ridge':
        model = Ridge()
        param_grid = {
            'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
            'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'saga']
        }
        n_iter = 30
    
    elif model_name == 'Lasso':
        model = Lasso(max_iter=5000)
        param_grid = {
            'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
            'selection': ['cyclic', 'random']
        }
        n_iter = 20
    
    elif model_name == 'ElasticNet':
        model = ElasticNet(max_iter=5000)
        param_grid = {
            'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
            'selection': ['cyclic', 'random']
        }
        n_iter = 30
    
    elif model_name == 'Neural Network':
        model = MLPRegressor(random_state=1065, early_stopping=True, max_iter=10000)
        param_grid = {
            'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100), (150, 100, 50)],
            'activation': ['relu', 'tanh'],
            'alpha': [0.0001, 0.001, 0.01],
            'learning_rate': ['constant', 'adaptive'],
            'learning_rate_init': [0.001, 0.01]
        }
        n_iter = 50
    
    else:
        raise ValueError(f"Model {model_name} not supported for tuning")
    
    # Randomized search with cross-validation
    rand_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=n_iter,
        cv=cv,
        scoring='r2',
        n_jobs=-1,
        random_state=1065,
        verbose=1
    )
    
    rand_search.fit(X_use, y)
    best_model = rand_search.best_estimator_
    
    # Get out-of-fold predictions using the best model
    oof_pred = cross_val_predict(best_model, X_use, y, cv=cv, n_jobs=-1)
    
    return best_model, rand_search.best_params_, oof_pred

def create_stacked_ensemble_oof(X, y, cv=5):
    # Base models
    base_models = [
        ('lgbm', LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, 
                               subsample=0.8, random_state=1065, verbose=-1)),
        ('xgb', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                             subsample=0.8, random_state=1065, tree_method='hist')),
        ('catboost', CatBoostRegressor(iterations=300, learning_rate=0.05, depth=5,
                                       random_state=1065, verbose=False)),
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=15, 
                                     random_state=1065, n_jobs=-1))
    ]
    
    # Meta-learner
    meta_model = Ridge(alpha=1.0)
    
    # Create stacking regressor with cross-validation built-in
    stacking_model = StackingRegressor(
        estimators=base_models,
        final_estimator=meta_model,
        cv=cv,
        n_jobs=-1
    )
    
    # Get out-of-fold predictions
    oof_pred = cross_val_predict(stacking_model, X, y, cv=cv, n_jobs=-1)
    
    # Calculate metrics
    oof_r2 = r2_score(y, oof_pred)
    oof_rmse = np.sqrt(mean_squared_error(y, oof_pred))
    oof_mae = mean_absolute_error(y, oof_pred)
    
    stacking_model.fit(X, y)
    return stacking_model, oof_pred


def comprehensive_model_analysis(X, y, dataset_name="Dataset", cv=5, scale_features=True, 
                                 models_to_tune='all', include_stacking=True, 
                                 save_plots=True, random_state=1065):
    # List of all available models
    all_models = [
        'LightGBM', 'XGBoost', 'CatBoost', 'Gradient Boosting',
        'Random Forest', 'Extra Trees', 'Ridge', 'Lasso', 
        'ElasticNet', 'Neural Network'
    ]
    
    # Determine which models to tune
    if models_to_tune == 'all':
        models_list = all_models
    elif isinstance(models_to_tune, list):
        models_list = models_to_tune
    else:
        raise ValueError("models_to_tune must be 'all' or a list of model names")
    
    tuned_results = []
    
    for model_name in models_list:
        try:
            tuned_model, best_params, tuned_oof_pred = hyperparameter_tune_model_oof(
                X, y, 
                model_name=model_name,
                cv=cv,
                scale_features=scale_features
            )
            
            tuned_results.append({
                'model_name': model_name,
                'tuned_model': tuned_model,
                'best_params': best_params,
                'oof_predictions': tuned_oof_pred,
                'oof_r2': r2_score(y, tuned_oof_pred),
                'oof_rmse': np.sqrt(mean_squared_error(y, tuned_oof_pred)),
                'oof_mae': mean_absolute_error(y, tuned_oof_pred)
            })
        except Exception as e:
            print(f"\nError tuning {model_name}: {e}")
            continue
    
    # 3. Stacked ensemble (optional)
    if include_stacking:
        print(f"\nStep 3: Creating stacked ensemble...")
        try:
            stacked_model, stacked_oof_pred = create_stacked_ensemble_oof(X, y, cv=cv)
            
            tuned_results.append({
                'model_name': 'Stacked Ensemble',
                'tuned_model': stacked_model,
                'best_params': None,
                'oof_predictions': stacked_oof_pred,
                'oof_r2': r2_score(y, stacked_oof_pred),
                'oof_rmse': np.sqrt(mean_squared_error(y, stacked_oof_pred)),
                'oof_mae': mean_absolute_error(y, stacked_oof_pred)
            })
        except Exception as e:
            print(f"\nError creating stacked ensemble: {e}")
    
    # Sort tuned results by OOF R²
    tuned_results_sorted = sorted(tuned_results, key=lambda x: x['oof_r2'], reverse=True)
    
    # Create parity plots for ALL TUNED MODELS
    n_tuned = len(tuned_results_sorted)
    n_cols = 3
    n_rows = int(np.ceil(n_tuned / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    axes = axes.flatten() if n_tuned > 1 else [axes]
    
    for idx, result in enumerate(tuned_results_sorted):
        parity_plot(y, result['oof_predictions'], 
                   f"Tuned {result['model_name']}", axes[idx])
    
    # Hide any unused subplots
    for idx in range(n_tuned, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 6. Feature importance from best tuned model
    best_tuned_model = tuned_results_sorted[0]['tuned_model']
    
    if hasattr(best_tuned_model, 'feature_importances_'):
        if hasattr(X, 'columns'):
            feature_names = X.columns
        else:
            feature_names = [f'Feature_{i}' for i in range(X.shape[1])]
        
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': best_tuned_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Plot feature importance
        plt.figure(figsize=(10, 8))
        plt.barh(feature_importance.head(20)['feature'], 
                 feature_importance.head(20)['importance'])
        plt.xlabel('Importance', fontsize=12, fontweight='bold')
        plt.title(f"Top 20 Feature Importances - {tuned_results_sorted[0]['model_name']} ({dataset_name})", 
                 fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
    else:
        feature_importance = None
        print(f"\nNote: {tuned_results_sorted[0]['model_name']} does not support feature importances")
    
    # Return comprehensive results
    return {
        'dataset_name': dataset_name,
        'initial_screening': initial_results,
        'tuned_results': tuned_results_sorted,
        'best_model': tuned_results_sorted[0]['tuned_model'],
        'best_model_name': tuned_results_sorted[0]['model_name'],
        'best_oof_r2': tuned_results_sorted[0]['oof_r2'],
        'feature_importance': feature_importance
    }

results_h = comprehensive_model_analysis(
    h_X, h_y, 
    dataset_name="H adsorbed",
    cv=5,
    scale_features=True,
    models_to_tune=[
        'LightGBM', 
        'XGBoost', 
        'CatBoost', 
        'Gradient Boosting',
        'Random Forest', 
        'Extra Trees', 
        'ElasticNet', 
        'Neural Network',
    ],
    include_stacking=True,
    save_plots=True
)

results_no_ads = comprehensive_model_analysis(
    no_ads_X, no_ads_y,
    dataset_name="No adsorbate", 
    cv=5,
    scale_features=True,
    models_to_tune=[
        'LightGBM', 
        'XGBoost', 
        'CatBoost', 
        'Gradient Boosting',
        'Random Forest', 
        'Extra Trees', 
        'ElasticNet', 
        'Neural Network',
    ],
    include_stacking=True,
    save_plots=True
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import uniform
import matplotlib.pyplot as plt

def fit_and_tune_model(X, y, model_type='lasso', dataset_name='', feature_names=None, n_iter=50, cv=5, random_state=42):
    # Generate feature names if not provided
    if feature_names is None:
        feature_names = [f"Feature_{i}" for i in range(X.shape[1])]
    
    # Define model and parameter distributions
    if model_type.lower() == 'lasso':
        model = Lasso(random_state=random_state, max_iter=10000)
        param_dist = {
            'alpha': uniform(0.0001, 10)  # Uniform distribution between 0.0001 and 10
        }
        model_name = 'Lasso'
    elif model_type.lower() == 'ridge':
        model = Ridge(random_state=random_state, max_iter=10000)
        param_dist = {
            'alpha': uniform(0.0001, 100)  # Uniform distribution between 0.0001 and 100
        }
        model_name = 'Ridge'
    else:
        raise ValueError("model_type must be 'lasso' or 'ridge'")
    
    print(f"\n{'='*60}")
    print(f"Tuning {model_name} Regression - {dataset_name}")
    print(f"{'='*60}")
    
    # Perform randomized search
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        random_state=random_state,
        n_jobs=-1,
        verbose=1
    )
    
    random_search.fit(X, y)
    
    # Get best model
    best_model = random_search.best_estimator_
    
    print(f"\nBest parameters: {random_search.best_params_}")
    print(f"Best CV score (neg_mse): {random_search.best_score_:.4f}")
    
    # Get out-of-fold predictions using best model
    oof_predictions = cross_val_predict(best_model, X, y, cv=cv)
    
    # Calculate OOF metrics
    oof_mse = mean_squared_error(y, oof_predictions)
    oof_rmse = np.sqrt(oof_mse)
    oof_mae = mean_absolute_error(y, oof_predictions)
    oof_r2 = r2_score(y, oof_predictions)
    
    # For Lasso, report number of non-zero coefficients
    if model_type.lower() == 'lasso':
        # Refit on full data to get final coefficients
        best_model.fit(X, y)
        n_features_selected = np.sum(best_model.coef_ != 0)
        print(f"\nFeatures selected: {n_features_selected} out of {X.shape[1]}")
    else:
        # Refit Ridge on full data as well
        best_model.fit(X, y)
    
    # Get coefficients
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': best_model.coef_,
        'Abs_Coefficient': np.abs(best_model.coef_)
    })
    
    # Sort by absolute value
    coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
    
    if model_type.lower() == 'lasso':
        # Show which features were eliminated
        eliminated_features = coef_df[coef_df['Coefficient'] == 0]
        if len(eliminated_features) > 0:
            print(f"\n{len(eliminated_features)} Features Eliminated by Lasso:")
            print(eliminated_features['Feature'].tolist())
    
    results = {
        'model': best_model,
        'best_params': random_search.best_params_,
        'oof_rmse': oof_rmse,
        'oof_mse': oof_mse,
        'oof_mae': oof_mae,
        'oof_r2': oof_r2,
        'oof_predictions': oof_predictions,
        'coefficients': coef_df
    }
    
    if model_type.lower() == 'lasso':
        results['n_features_selected'] = n_features_selected
    
    return best_model, results


def plot_coefficient_comparison(results_lasso, results_ridge, dataset_name, top_n=15):    
    # Get coefficients
    lasso_coef = results_lasso['coefficients'].copy()
    ridge_coef = results_ridge['coefficients'].copy()
    
    # Merge on feature name
    comparison = pd.merge(
        lasso_coef[['Feature', 'Coefficient']], 
        ridge_coef[['Feature', 'Coefficient']], 
        on='Feature', 
        suffixes=('_Lasso', '_Ridge')
    )
    
    # Add absolute values for sorting
    comparison['Abs_Ridge'] = np.abs(comparison['Coefficient_Ridge'])
    comparison = comparison.sort_values('Abs_Ridge', ascending=False).head(top_n)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(8, 6))
    
    x = np.arange(len(comparison))
    width = 0.35
    
    ax.barh(x - width/2, comparison['Coefficient_Lasso'], width, label='Lasso', alpha=0.8)
    ax.barh(x + width/2, comparison['Coefficient_Ridge'], width, label='Ridge', alpha=0.8)
    
    ax.set_yticks(x)
    ax.set_yticklabels(comparison['Feature'])
    ax.set_xlabel('Coefficient Value')
    ax.set_title(f'Top {top_n} Feature Coefficients: Lasso vs Ridge - {dataset_name}')
    ax.legend()
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return comparison

def plot_all_parity_plots(results_dict, y_dict):
    # Create a 2x2 grid of subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 16))
    axes = axes.flatten()
    
    plot_idx = 0
    for (model_name, dataset_name), results in results_dict.items():
        y_true = y_dict[dataset_name]
        y_pred = results['oof_predictions']
        r2 = results['oof_r2']
        rmse = results['oof_rmse']
        mae = results['oof_mae']
        
        ax = axes[plot_idx]
        
        # Scatter plot
        ax.scatter(y_true, y_pred, alpha=0.5, s=30, edgecolors='k', linewidths=0.5)
        
        # Perfect prediction line
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
        
        # Add text box with metrics
        textstr = f'R² = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}'
        props = dict(boxstyle='round', facecolor='white', alpha=0.8)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=props)
        
        # Labels and title
        ax.set_xlabel('Actual Values', fontsize=12)
        ax.set_ylabel('Predicted Values', fontsize=12)
        ax.set_title(f'{model_name} - {dataset_name}', 
                     fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.set_aspect('equal', adjustable='box')
        
        plot_idx += 1
    
    plt.tight_layout()
    plt.show()

feature_names = list(h_X.columns) if hasattr(h_X, 'columns') else None

# Fit Lasso models
lasso_h, results_lasso_h = fit_and_tune_model(
    h_X, h_y, 
    model_type='lasso', 
    dataset_name='h dataset',
    feature_names=feature_names,  # Replace with actual feature names if available
    n_iter=50,
    cv=5
)

lasso_no_ads, results_lasso_no_ads = fit_and_tune_model(
    no_ads_X, no_ads_y, 
    model_type='lasso', 
    dataset_name='no_ads dataset',
    feature_names=feature_names,  # Replace with actual feature names if available
    n_iter=50,
    cv=5
)

# Fit Ridge models
ridge_h, results_ridge_h = fit_and_tune_model(
    h_X, h_y, 
    model_type='ridge', 
    dataset_name='h dataset',
    feature_names=feature_names,  # Replace with actual feature names if available
    n_iter=50,
    cv=5
)

ridge_no_ads, results_ridge_no_ads = fit_and_tune_model(
    no_ads_X, no_ads_y, 
    model_type='ridge', 
    dataset_name='no_ads dataset',
    feature_names=feature_names,  # Replace with actual feature names if available
    n_iter=50,
    cv=5
)

plot_coefficient_comparison(results_lasso_h, results_ridge_h, 'h dataset', top_n=15)
plot_coefficient_comparison(results_lasso_no_ads, results_ridge_no_ads, 'no_ads dataset', top_n=15)

# Prepare dictionaries for batch plotting
results_dict = {
    ('Lasso', 'h'): results_lasso_h,
    ('Lasso', 'no_ads'): results_lasso_no_ads,
    ('Ridge', 'h'): results_ridge_h,
    ('Ridge', 'no_ads'): results_ridge_no_ads
}

y_dict = {
    'h': h_y,
    'no_ads': no_ads_y
}

plot_all_parity_plots(results_dict, y_dict)